In [1]:
# #!/usr/bin/env python3
# """
# Versão minimalista para baixar train.bin
# """

# import os
# from huggingface_hub import hf_hub_download

# # Configure aqui
# REPO_ID = "marcos-j-leemes/tinyS"
# TOKEN = "SEU_TOKEN_AQUI"  # Coloque seu token aqui | acelera o downloading
# FILENAME = "tinystories_tokens.bin"

# # Baixa
# print(f"Baixando {FILENAME}...")
# path = hf_hub_download(
#     repo_id=REPO_ID,
#     filename=FILENAME,
#     repo_type="dataset",
#     token=TOKEN,
#     local_dir=".",
# )

# print(f" Baixado para: {path}")

Baixando tinystories_tokens.bin...


tinystories_tokens.bin: reconstructing file:   0%|          |  0.00B /  944MB            

tinystories_tokens.bin: downloading bytes:           |  0.00B            

 Baixado para: /kaggle/working/tinystories_tokens.bin


In [16]:
"""GPT com Pipeline Parallelism (GPipe) para N GPUs, treino de 10 minutos.

Uso (Kaggle, 2 x T4):
    !torchrun --standalone --nproc_per_node=2 train_pp.py

O número de stages é igual ao número de GPUs (nproc_per_node). O modelo é
particionado automaticamente por custo de computação (ver partition_layers),
então mudar --nproc_per_node já redistribui as camadas.

Arquivo de dados: tokens GPT-2 em uint16 (tinystories_tokens.bin). Pode apontar
outro caminho com a variável de ambiente DATA_PATH.

torch.compile: ligado por padrão. Desligue com USE_COMPILE=0 para depurar.
O checkpoint é sempre salvo a partir do módulo NÃO compilado.


Para 20 minutos em duas T4:

C = 65×10¹² × 0,1 × 1200 × 2 = 1,56×10¹⁶ FLOPs
N = √(C/120) = √(1,56×10¹⁶ / 120) = √(1,3×10¹⁴) ≈ 11,4 M parâmetros
D = 20·N ≈ 228 M tokens

+1 min: 7,8×10^14   FLOPs (+5%)

Para suprir essa previsão será necessário aumentar 1 minuto de tempo de treino
O que equivale a mais 5% de computação
Teste: 

"""

import math
import os
import time
from datetime import timedelta

import numpy as np
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F

# ============================== CONFIGURAÇÃO ==============================
DATA_PATH = os.environ.get("DATA_PATH", "tinystories_tokens.bin")

# torch.compile (desligue com USE_COMPILE=0 para depurar)
USE_COMPILE = os.environ.get("USE_COMPILE", "1") == "1"

# Modelo (~41M parâmetros, 19M deles no embedding/lm_head amarrados)
VOCAB_SIZE = 50_257
EMBEDDING_DIM = 128
NUM_HEADS = 2              # head_dim = 64
NUM_LAYERS = 2
BLOCK_SIZE = 450           # contexto; TinyStories são histórias curtas
DROPOUT = 0.0

# Orçamento de tempo: o treino para sozinho e o LR decai em função do tempo
TIME_BUDGET = 1260           # segundos de treino (sem contar setup e checkpoint)
# 21 minuto



# Batch: tokens/step = BATCH_SIZE * NUM_MICROBATCHES * BLOCK_SIZE = 32.768
# Em GPipe, os microbatches SÃO a gradient accumulation (grad_acc = 8).
# Bolha do pipeline com 2 stages = 1/(M+1) = ~11%.
BATCH_SIZE = 18            # sequências por microbatch
NUM_MICROBATCHES = 8 #8       # grad_acc

# Otimizador
LR = 1.5e-3
MIN_LR = 1.5e-4
WARMUP_STEPS = 30
WEIGHT_DECAY = 0.1
BETAS = (0.9, 0.95)
GRAD_CLIP = 1.0

# Particionamento. None = automático (balanceia custo entre os stages).
# Ou defina manualmente blocos por stage, ex.: PARTITION = [10, 2]
PARTITION = [1, 1]
# O lm_head (50k de vocabulário) + softmax/CE custa bem mais que um bloco;
# este fator corrige o custo estimado do head no balanceamento automático.
HEAD_COST_FACTOR = 1.5

# fp16 (T4 não tem bf16) com loss scaling dinâmico sincronizado entre ranks
INIT_LOSS_SCALE = 2.0 ** 12
SCALE_GROWTH_INTERVAL = 200

PRINT_INTERVAL = 10
SAVE_PREFIX = "model_final"
SEED = 1337

CONFIG = {
    "vocab_size": VOCAB_SIZE,
    "embedding_dim": EMBEDDING_DIM,
    "num_heads": NUM_HEADS,
    "num_layers": NUM_LAYERS,
    "block_size": BLOCK_SIZE,
    "dropout": DROPOUT,
}
# ==========================================================================


# ------------------------------- Modelo -----------------------------------
class Block(nn.Module):
    def __init__(self, d, h, dropout):
        super().__init__()
        self.h, self.dropout = h, dropout
        self.ln1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.proj = nn.Linear(d, d, bias=False)
        self.ln2 = nn.LayerNorm(d)
        self.fc = nn.Linear(d, 4 * d, bias=False)
        self.out = nn.Linear(4 * d, d, bias=False)
        self.proj._res = True
        self.out._res = True

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(C, dim=2)
        q, k, v = (t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v))
        a = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0
        )
        x = x + self.proj(a.transpose(1, 2).reshape(B, T, C))
        return x + self.out(F.gelu(self.fc(self.ln2(x))))


class GPT(nn.Module):
    """Modelo lógico completo (usado para inicializar, particionar e amostrar)."""

    def __init__(self, cfg):
        super().__init__()
        d = cfg["embedding_dim"]
        self.n_layer = cfg["num_layers"]
        self.wte = nn.Embedding(cfg["vocab_size"], d)
        self.wpe = nn.Embedding(cfg["block_size"], d)
        self.blocks = nn.ModuleList(
            Block(d, cfg["num_heads"], cfg["dropout"]) for _ in range(self.n_layer)
        )
        self.ln_f = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.wte.weight  # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            std = 0.02 / math.sqrt(2 * self.n_layer) if getattr(m, "_res", False) else 0.02
            nn.init.normal_(m.weight, 0.0, std)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, ids):
        pos = torch.arange(ids.size(1), device=ids.device)
        h = self.wte(ids) + self.wpe(pos)
        for b in self.blocks:
            h = b(h)
        return self.lm_head(self.ln_f(h))


class Stage(nn.Module):
    """Um pedaço contíguo do modelo: [embedding?] + blocos[lo:hi] + [ln_f + lm_head?]."""

    def __init__(self, full, lo, hi, first, last):
        super().__init__()
        self.first, self.last, self.block_lo = first, last, lo
        if first:
            self.wte, self.wpe = full.wte, full.wpe
        self.blocks = nn.ModuleList(list(full.blocks)[lo:hi])
        if last:
            self.ln_f = full.ln_f
            self.lm_head = nn.Linear(full.lm_head.in_features, full.lm_head.out_features, bias=False)
            if first:  # GPU única: tying real
                self.lm_head.weight = self.wte.weight
            else:      # cópia do peso do wte; gradientes somados a cada step
                with torch.no_grad():
                    self.lm_head.weight.copy_(full.lm_head.weight)

    def forward(self, inp, targets=None):
        if self.first:
            pos = torch.arange(inp.size(1), device=inp.device)
            h = self.wte(inp) + self.wpe(pos)
        else:
            h = inp
        for b in self.blocks:
            h = b(h)
        if not self.last:
            return h
        logits = self.lm_head(self.ln_f(h))
        return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.view(-1))


def partition_layers(world):
    """Retorna quantos blocos cada stage recebe (lista de tamanho `world`).

    Particiona [embedding, bloco_0..bloco_{L-1}, head] em `world` grupos
    contíguos minimizando o custo do grupo mais pesado (programação dinâmica).
    Custo por token (MACs): bloco = 12*D^2 + 2*T*D ; head = fator * D * V.
    """
    L, D, T, V = NUM_LAYERS, EMBEDDING_DIM, BLOCK_SIZE, VOCAB_SIZE
    if PARTITION is not None:
        if len(PARTITION) != world or sum(PARTITION) != L:
            raise ValueError(f"PARTITION deve ter {world} itens somando {L}.")
        return list(PARTITION)

    costs = [0.0] + [12 * D * D + 2 * T * D] * L + [HEAD_COST_FACTOR * D * V]
    n = len(costs)
    if world > n:
        raise ValueError("Mais GPUs do que itens para particionar.")
    pre = [0.0]
    for c in costs:
        pre.append(pre[-1] + c)

    INF = float("inf")
    dp = [[INF] * (n + 1) for _ in range(world + 1)]
    cut = [[0] * (n + 1) for _ in range(world + 1)]
    dp[0][0] = 0.0
    for k in range(1, world + 1):
        for i in range(k, n + 1):
            for j in range(k - 1, i):
                c = max(dp[k - 1][j], pre[i] - pre[j])
                if c < dp[k][i]:
                    dp[k][i], cut[k][i] = c, j
    bounds, i = [], n
    for k in range(world, 0, -1):
        j = cut[k][i]
        bounds.append((j, i))
        i = j
    bounds.reverse()
    # item 0 = embedding, itens 1..L = blocos, item L+1 = head
    return [max(0, min(b, L + 1) - max(a, 1)) for a, b in bounds]


# ------------------------------- Dados ------------------------------------
class TokenBatcher:
    """Amostragem determinística: ranks diferentes geram o MESMO batch para
    (step, microbatch), então o stage 0 usa x e o último stage usa y, sem
    precisar enviar targets pela rede."""

    def __init__(self, path):
        if not os.path.exists(path):
            raise FileNotFoundError(f"Dataset não encontrado: {path}")
        self.data = np.memmap(path, dtype=np.uint16, mode="r")
        if len(self.data) <= BLOCK_SIZE + 1:
            raise ValueError("Dataset com tokens insuficientes.")
        self.offsets = np.arange(BLOCK_SIZE + 1)

    def get(self, step, m, device, need_x, need_y):
        rng = np.random.default_rng([SEED, step, m])
        starts = rng.integers(0, len(self.data) - BLOCK_SIZE - 1, size=BATCH_SIZE)
        chunk = self.data[starts[:, None] + self.offsets].astype(np.int64)
        if chunk.max() >= VOCAB_SIZE:
            raise ValueError(f"Token fora do vocabulário (max={chunk.max()}, VOCAB_SIZE={VOCAB_SIZE}).")
        t = torch.from_numpy(chunk)
        x = t[:, :-1].contiguous().to(device) if need_x else None
        y = t[:, 1:].contiguous().to(device) if need_y else None
        return x, y


# ------------------------------ Pipeline ----------------------------------
def pipeline_step(model, batcher, device, rank, world, step, scale):
    """Um step GPipe: M microbatches forward -> backward, gradiente acumulado.

    `model` aqui pode ser o módulo compilado (só é usado para chamar forward).

    - Último stage: forward+backward imediato por microbatch (libera os logits
      de 50k de vocabulário cedo e começa a devolver gradientes logo).
    - Demais stages: todos os forwards, depois backwards na ordem em que os
      gradientes chegam.
    Ordem das operações P2P por par de ranks é sempre a mesma nos dois lados
    (envia ativações -> recebe grads / recebe ativações -> envia grads), o que
    evita deadlock em NCCL.
    """
    M = NUM_MICROBATCHES
    first, last = rank == 0, rank == world - 1
    shape = (BATCH_SIZE, BLOCK_SIZE, EMBEDDING_DIM)
    works, keep = [], []  # mantém tensores vivos até o fim das isend

    in_bufs, in_works = [], []
    if not first:  # posta todos os recv de ativação de uma vez
        for _ in range(M):
            buf = torch.empty(shape, device=device, dtype=torch.float32)
            in_bufs.append(buf)
            in_works.append(dist.irecv(buf, src=rank - 1))

    def next_input(m):
        if first:
            x, _ = batcher.get(step, m, device, need_x=True, need_y=last)
            return x, None
        in_works[m].wait()
        return in_bufs[m].requires_grad_(True), None

    loss_acc = torch.zeros((), device=device) if last else None

    if last:
        for m in range(M):
            inp, _ = next_input(m)
            _, y = batcher.get(step, m, device, need_x=False, need_y=True)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(inp, y)
            (loss * (scale / M)).backward()
            loss_acc += loss.detach() / M
            if not first:
                works.append(dist.isend(inp.grad, dst=rank - 1))
                keep.append(inp.grad)
    else:
        ins, outs = [], []
        for m in range(M):
            inp, _ = next_input(m)
            with torch.autocast("cuda", dtype=torch.float16):
                out = model(inp)
            od = out.detach()
            works.append(dist.isend(od, dst=rank + 1))
            keep.append(od)
            ins.append(inp)
            outs.append(out)
        grads = [torch.empty(shape, device=device, dtype=torch.float32) for _ in range(M)]
        gworks = [dist.irecv(g, src=rank + 1) for g in grads]
        for m in range(M):
            gworks[m].wait()
            outs[m].backward(grads[m])
            if not first:
                works.append(dist.isend(ins[m].grad, dst=rank - 1))
                keep.append(ins[m].grad)
            outs[m] = None

    for w in works:
        w.wait()
    return loss_acc


def get_lr(step, progress):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    return MIN_LR + 0.5 * (LR - MIN_LR) * (1 + math.cos(math.pi * min(progress, 1.0)))


# ------------------------------ Checkpoint --------------------------------
def save_stage(model, rank):
    """Salva o state_dict do módulo NÃO compilado (chaves globais, sem _orig_mod)."""
    sd = {}
    for k, v in model.state_dict().items():
        k = k.replace("_orig_mod.", "")  # defesa: nunca grava prefixo do compile
        if k.startswith("blocks."):  # nomes globais: blocks.<idx global>...
            parts = k.split(".")
            parts[1] = str(int(parts[1]) + model.block_lo)
            k = ".".join(parts)
        sd[k] = v.detach().cpu()
    path = f"{SAVE_PREFIX}_stage{rank}.pth"
    torch.save({"stage": rank, "config": CONFIG, "state_dict": sd}, path)
    print(f"[rank {rank}] checkpoint salvo em {path}", flush=True)


@torch.no_grad()
def merge_and_sample(world, device):
    sd = {}
    for r in range(world):
        sd.update(torch.load(f"{SAVE_PREFIX}_stage{r}.pth", map_location="cpu")["state_dict"])
    torch.save({"config": CONFIG, "state_dict": sd}, f"{SAVE_PREFIX}.pth")
    print(f"modelo completo salvo em {SAVE_PREFIX}.pth", flush=True)
    try:
        import tiktoken
    except ImportError:
        print("tiktoken não instalado: pulando amostra de texto.", flush=True)
        return
    model = GPT(CONFIG).to(device).eval()
    model.load_state_dict(sd)
    enc = tiktoken.get_encoding("gpt2")
    ids = torch.tensor([enc.encode("Once upon a time")], device=device)
    for _ in range(120):
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(ids[:, -BLOCK_SIZE:])[:, -1].float() / 0.8
        v, _ = torch.topk(logits, 40)
        logits[logits < v[:, [-1]]] = -float("inf")
        ids = torch.cat([ids, torch.multinomial(F.softmax(logits, -1), 1)], dim=1)
    print("\n--- amostra ---\n" + enc.decode(ids[0].tolist()) + "\n", flush=True)


# -------------------------------- Main ------------------------------------
def main():
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    dist.init_process_group(backend="nccl", timeout=timedelta(minutes=30))
    rank, world = dist.get_rank(), dist.get_world_size()
    device = torch.device(f"cuda:{local_rank}")
    first, last = rank == 0, rank == world - 1

    try:
        # Mesma inicialização em todos os ranks; cada um guarda só sua parte.
        torch.manual_seed(SEED)
        torch.cuda.manual_seed(SEED)
        counts = partition_layers(world)
        lo = sum(counts[:rank])
        full = GPT(CONFIG)
        model = Stage(full, lo, lo + counts[rank], first, last).to(device)
        del full
        model.train()

        tied_group = dist.new_group([0, world - 1]) if world > 1 else None
        tied_copy = model.lm_head.weight if (last and not first) else None
        norm_params = [p for p in model.parameters() if p is not tied_copy]

        decay = [p for p in model.parameters() if p.ndim >= 2]
        no_decay = [p for p in model.parameters() if p.ndim < 2]
        optimizer = torch.optim.AdamW(
            [{"params": decay, "weight_decay": WEIGHT_DECAY},
             {"params": no_decay, "weight_decay": 0.0}],
            lr=LR, betas=BETAS, eps=1e-8, fused=True,
        )
        batcher = TokenBatcher(DATA_PATH) if (first or last) else None

        # Módulo compilado: COMPARTILHA os mesmos parâmetros de `model`.
        # Otimizador, clip e checkpoint continuam usando `model` (não compilado).
        cmodel = torch.compile(model, dynamic=False) if USE_COMPILE else model

        n_params = sum(p.numel() for p in model.parameters())
        print(f"[rank {rank}] blocos {lo}..{lo + counts[rank] - 1 if counts[rank] else lo - 1} "
              f"({counts[rank]}) | embed={first} head={last} | {n_params / 1e6:.1f}M params",
              flush=True)
        if first:
            tps = BATCH_SIZE * BLOCK_SIZE * NUM_MICROBATCHES
            print(f"GPUs={world} | partição (blocos/stage)={counts} | "
                  f"tokens/step={tps:,} | microbatches={NUM_MICROBATCHES} | "
                  f"compile={USE_COMPILE} | orçamento={TIME_BUDGET}s", flush=True)

        tokens_per_step = BATCH_SIZE * BLOCK_SIZE * NUM_MICROBATCHES
        scale, good_steps, step, tokens_total = INIT_LOSS_SCALE, 0, 0, 0
        ctrl = torch.zeros(1, device=device)

        # Warmup: dispara a compilação (forward + backward) FORA do orçamento de
        # tempo. Não chama optimizer.step(), então os pesos não mudam.
        if USE_COMPILE:
            if first:
                print("compilando (warmup)...", flush=True)
            tc = time.perf_counter()
            pipeline_step(cmodel, batcher, device, rank, world, 0, scale)
            optimizer.zero_grad(set_to_none=True)
            torch.cuda.synchronize(device)
            print(f"[rank {rank}] compilação: {time.perf_counter() - tc:.0f}s", flush=True)

        dist.barrier()
        t0 = time.perf_counter()

        while True:
            # rank 0 decide o tempo decorrido; todos obedecem (LR e parada iguais)
            if first:
                ctrl[0] = time.perf_counter() - t0
            dist.broadcast(ctrl, src=0)
            elapsed = ctrl.item()
            if elapsed >= TIME_BUDGET:
                break
            lr = get_lr(step, elapsed / TIME_BUDGET)
            for g in optimizer.param_groups:
                g["lr"] = lr

            step_start = time.perf_counter()
            optimizer.zero_grad(set_to_none=True)
            loss_acc = pipeline_step(cmodel, batcher, device, rank, world, step, scale)

            # weight tying lógico: soma grad do wte (rank 0) e do lm_head (último)
            if world > 1 and (first or last):
                g = model.wte.weight.grad if first else model.lm_head.weight.grad
                dist.all_reduce(g, group=tied_group)

            # norma global (todos os stages); a cópia do lm_head não é contada 2x
            all_grads = [p.grad for p in model.parameters() if p.grad is not None]
            sq = torch.stack(torch._foreach_norm([p.grad for p in norm_params])).float().pow(2).sum()
            if world > 1:
                dist.all_reduce(sq)
            total_norm = sq.sqrt()
            finite = bool(torch.isfinite(total_norm).item())  # igual em todos os ranks
            norm_true = total_norm.item() / scale if finite else float("inf")

            if finite:
                coef = min(1.0, GRAD_CLIP / (norm_true + 1e-6))
                torch._foreach_mul_(all_grads, coef / scale)  # desfaz scale + clip
                optimizer.step()
                good_steps += 1
                if good_steps % SCALE_GROWTH_INTERVAL == 0:
                    scale *= 2.0
            else:
                scale = max(scale / 2.0, 1.0)
                good_steps = 0

            tokens_total += tokens_per_step
            if last and step % PRINT_INTERVAL == 0:
                dt = time.perf_counter() - step_start
                print(f"step {step:04d} | loss {loss_acc.item():.4f} | lr {lr:.2e} | "
                      f"gnorm {norm_true:.3f} | {tokens_per_step / dt:,.0f} tok/s | "
                      f"tokens {tokens_total:,} | {elapsed:5.0f}/{TIME_BUDGET}s | "
                      f"scale {scale:.0f}{'' if finite else ' | PULADO (overflow)'}",
                      flush=True)
            step += 1

        torch.cuda.synchronize(device)
        total = time.perf_counter() - t0
        if last:
            print(f"\nFim: {step} steps, {tokens_total:,} tokens em {total:.0f}s "
                  f"({tokens_total / total:,.0f} tok/s médio)", flush=True)
        print(f"[rank {rank}] pico de memória: "
              f"{torch.cuda.max_memory_allocated(device) / 2**30:.1f} GiB", flush=True)

        # Salva sempre o módulo original (não compilado)
        save_stage(model, rank)
        dist.barrier()
        if first:
            merge_and_sample(world, device)
        dist.barrier()
    finally:
        if dist.is_available() and dist.is_initialized():
            dist.destroy_process_group()


if __name__ == "__main__":

    inicio = time.time()
    main()

    print()
    print("="*80)
    fim = time.time()
    minuto = (fim - inicio) / 60
    print(f"Tempo total de execução: {fim - inicio:.2f} segundos")
    print(f"Tempo total de execução: {minuto:.2f} minutos")
    print("="*80)
    print()
    
    exit()

KeyError: 'LOCAL_RANK'

In [29]:
!torchrun --nproc_per_node=2 /kaggle/working/.virtual_documents/__notebook_source__.ipynb

W1008 16:48:47.781000 1262 torch/distributed/run.py:851] 
W1008 16:48:47.781000 1262 torch/distributed/run.py:851] *****************************************
W1008 16:48:47.781000 1262 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1008 16:48:47.781000 1262 torch/distributed/run.py:851] *****************************************
[rank 1] blocos 1..1 (1) | embed=False head=True | 6.6M params
[rank1]:[W1008 16:48:52.611375902 ProcessGroupNCCL.cpp:4094] Warning: An unbatched P2P op (send/recv) was called on this ProcessGroup with size 2.  In lazy initialization mode, this will result in a new 2-rank NCCL communicator to be created. (function operator())
[rank 0] blocos 0..0 (1) | embed=True head=False | 6.7M params
GPUs=2 | partição (blocos/stage)=[1, 1] | tokens/step=64,800 | microbatches=8 | compil

In [21]:
# 20 mintuos


# Fim: 3366 steps, 218,116,800 tokens em 1200s (181,731 tok/s médio)
# [rank 0] pico de memória: 0.5 GiB
# [rank 1] pico de memória: 1.8 GiB
# [rank 0] checkpoint salvo em model_final_stage0.pth
# [rank 1] checkpoint salvo em model_final_stage1.pth
# modelo completo salvo em model_final.pth

# --- amostra ---
# Once upon a time there was an old lady. She was always very kind to her family in the house. One day, she asked her mom if she could have a special gift from the store. So they went to her grandma's house. 

# At the store, Lily saw a small store. She was so excited that she grabbed the box with the box. She said she would pick the box inside. It was a nice girl's grandma with a lot of money. 

# The lady was so happy she went to the store and bought some new things. Lily was so happy and she thanked her


# ================================================================================
# Tempo total de execução: 1211.33 segundos
# Tempo total de execução: 20.19 minutos
# ================================================================================

#!torchrun --nproc_per_node=2 /kaggle/working/.virtual_documents/__notebook_source__.ipynb

W1008 16:22:26.285000 1074 torch/distributed/run.py:851] 
W1008 16:22:26.285000 1074 torch/distributed/run.py:851] *****************************************
W1008 16:22:26.285000 1074 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1008 16:22:26.285000 1074 torch/distributed/run.py:851] *****************************************
[rank 0] blocos 0..0 (1) | embed=True head=False | 6.7M params
GPUs=2 | partição (blocos/stage)=[1, 1] | tokens/step=64,800 | microbatches=8 | compile=True | orçamento=1200s
compilando (warmup)...
[rank 1] blocos 1..1 (1) | embed=False head=True | 6.6M params
[rank1]:[W1008 16:22:30.910699537 ProcessGroupNCCL.cpp:4094] Warning: An unbatched P2P op (send/recv) was called on this ProcessGroup with size 2.  In lazy initialization mode, this will result in a new 2-rank NCCL co

In [2]:
!ls

tinystories_tokens.bin


In [36]:
import math
from pathlib import Path

import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F


class Block(nn.Module):
    def __init__(self, d, h, dropout):
        super().__init__()
        self.h, self.dropout = h, dropout
        self.ln1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.proj = nn.Linear(d, d, bias=False)
        self.ln2 = nn.LayerNorm(d)
        self.fc = nn.Linear(d, 4 * d, bias=False)
        self.out = nn.Linear(4 * d, d, bias=False)
        self.proj._res = True
        self.out._res = True

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(C, dim=2)
        q, k, v = (t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v))
        a = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0
        )
        x = x + self.proj(a.transpose(1, 2).reshape(B, T, C))
        return x + self.out(F.gelu(self.fc(self.ln2(x))))


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg["embedding_dim"]
        self.n_layer = cfg["num_layers"]
        self.wte = nn.Embedding(cfg["vocab_size"], d)
        self.wpe = nn.Embedding(cfg["block_size"], d)
        self.blocks = nn.ModuleList(
            Block(d, cfg["num_heads"], cfg["dropout"]) for _ in range(self.n_layer)
        )
        self.ln_f = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.wte.weight
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            std = 0.02 / math.sqrt(2 * self.n_layer) if getattr(m, "_res", False) else 0.02
            nn.init.normal_(m.weight, 0.0, std)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, ids):
        pos = torch.arange(ids.size(1), device=ids.device)
        h = self.wte(ids) + self.wpe(pos)
        for b in self.blocks:
            h = b(h)
        return self.lm_head(self.ln_f(h))


class Stage(nn.Module):
    def __init__(self, full, lo, hi, first, last):
        super().__init__()
        self.first, self.last, self.block_lo = first, last, lo
        if first:
            self.wte, self.wpe = full.wte, full.wpe
        self.blocks = nn.ModuleList(list(full.blocks)[lo:hi])
        if last:
            self.ln_f = full.ln_f
            self.lm_head = nn.Linear(full.lm_head.in_features, full.lm_head.out_features, bias=False)
            if first:
                self.lm_head.weight = self.wte.weight
            else:
                with torch.no_grad():
                    self.lm_head.weight.copy_(full.lm_head.weight)

    def forward(self, inp, targets=None):
        if self.first:
            pos = torch.arange(inp.size(1), device=inp.device)
            h = self.wte(inp) + self.wpe(pos)
        else:
            h = inp
        for b in self.blocks:
            h = b(h)
        if not self.last:
            return h
        logits = self.lm_head(self.ln_f(h))
        return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.view(-1))


class InferStage(Stage):
    def forward(self, inp):
        if self.first:
            pos = torch.arange(inp.size(1), device=inp.device)
            h = self.wte(inp) + self.wpe(pos)
        else:
            h = inp
        for block in self.blocks:
            h = block(h)
        if not self.last:
            return h
        return self.lm_head(self.ln_f(h[:, -1:, :]))[:, 0, :]


CHECKPOINT_PREFIX = "model_final"
CHECKPOINT_DIR = Path(".")


def load_pipeline():
    files = sorted(
        CHECKPOINT_DIR.glob(f"{CHECKPOINT_PREFIX}_stage*.pth"),
        key=lambda p: int(p.stem.split("stage")[-1]),
    )
    if not files:
        raise FileNotFoundError(f"Nenhum {CHECKPOINT_PREFIX}_stage*.pth em {CHECKPOINT_DIR.resolve()}")
    ckpts = [torch.load(f, map_location="cpu") for f in files]
    for rank, ck in enumerate(ckpts):
        if ck["stage"] != rank:
            raise ValueError(f"Stage faltando ou fora de ordem: esperado {rank}, veio {ck['stage']}.")

    config = ckpts[0]["config"]
    n = len(ckpts)

    full = GPT(config)
    stages, lo = [], 0
    for rank, ck in enumerate(ckpts):
        idx = {int(k.split(".")[1]) for k in ck["state_dict"] if k.startswith("blocks.")}
        hi = lo + len(idx)
        if idx and (min(idx) != lo or max(idx) != hi - 1):
            raise ValueError(f"Blocos do stage {rank} não são contíguos a partir de {lo}.")
        stages.append(InferStage(full, lo, hi, first=rank == 0, last=rank == n - 1))
        lo = hi
    if lo != config["num_layers"]:
        raise ValueError(f"Checkpoints cobrem {lo} blocos, esperado {config['num_layers']}.")
    del full

    for stage, ck in zip(stages, ckpts):
        sd = {}
        for k, v in ck["state_dict"].items():
            if k.startswith("blocks."):
                parts = k.split(".")
                parts[1] = str(int(parts[1]) - stage.block_lo)
                k = ".".join(parts)
            sd[k] = v
        stage.load_state_dict(sd, strict=True)

    n_gpu = torch.cuda.device_count()
    devices = [
        torch.device(f"cuda:{i if n_gpu >= n else i % n_gpu}") if n_gpu else torch.device("cpu")
        for i in range(n)
    ]
    stages = [s.to(d).eval() for s, d in zip(stages, devices)]
    for i, (s, d) in enumerate(zip(stages, devices)):
        print(f"Stage {i}: {len(s.blocks)} bloco(s) em {d}")
    return stages, devices, config


@torch.inference_mode()
def generate(stages, devices, input_ids, block_size, max_new_tokens=100,
             temperature=1.0, top_k=None):
    if temperature <= 0:
        raise ValueError("temperature deve ser maior que zero.")
    use_amp = devices[0].type == "cuda"

    ids = input_ids.to(devices[0])
    for _ in range(max_new_tokens):
        h = ids[:, -block_size:]
        with torch.autocast(devices[0].type, dtype=torch.float16, enabled=use_amp):
            for stage, device in zip(stages, devices):
                h = stage(h.to(device))
        next_logits = h.float() / temperature

        if top_k is not None:
            k = min(int(top_k), next_logits.size(-1))
            cutoff = torch.topk(next_logits, k).values[:, [-1]]
            next_logits = next_logits.masked_fill(next_logits < cutoff, float("-inf"))

        next_token = torch.multinomial(F.softmax(next_logits, dim=-1), num_samples=1)
        ids = torch.cat((ids, next_token.to(devices[0])), dim=1)
    return ids


prompt = "One day, A big girl"
max_new_tokens = 450
temperature = 0.7
top_k = 40

stages, devices, config = load_pipeline()
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = tokenizer.encode(prompt)
if not token_ids:
    raise ValueError("O prompt não gerou tokens.")
if max(token_ids) >= config["vocab_size"]:
    raise ValueError("O tokenizer produziu um ID fora de vocab_size.")

output = generate(
    stages, devices, torch.tensor([token_ids], dtype=torch.long),
    config["block_size"], max_new_tokens, temperature, top_k,
)
print("\nGenerated text:\n", tokenizer.decode(output[0].tolist()))

Stage 0: 1 bloco(s) em cuda:0
Stage 1: 1 bloco(s) em cuda:1

Generated text:
 One day, A big girl named Lily went to the store to buy her favorite toy. She saw a big box with a big lid. She opened it and looked inside. She saw a little girl sitting on a chair. The girl was very sad. She wanted to see what was inside. She looked inside and saw a little girl. She had a lot of trouble. She was very worried.

Lily saw a man who had been looking for a long time. She saw her and said, "I'm sorry, Lily. I was looking for the present." She said, "Don't worry, I will help you next time. I will take you out of here." She picked up the package and put it in the box. She said, "OK, Lily. I will help you. You can be my friend."

Lily felt a little happy because she was happy. She hugged her mom and made a new friend. She said, "Thank you, Lily. You are very helpful. You are a good friend. I love you."

The moral of the story is: Do not take things that are not yours. It's a story, and that the fami

In [37]:
from pathlib import Path
import torch

CHECKPOINT_PREFIX = "model_final"
CHECKPOINT_DIR = Path(".")


def count_parameters_from_state_dict(state_dict):
    """Conta os parâmetros diretamente a partir do state_dict."""
    total = 0

    for name, tensor in state_dict.items():
        if isinstance(tensor, torch.Tensor):
            # Ignora buffers que não são parâmetros
            if not any(x in name.lower() for x in [
                "running_mean",
                "running_var",
                "num_batches_tracked",
            ]):
                total += tensor.numel()

    return total


def get_checkpoint_info():
    # Aceita .pth e .pt
    checkpoints = sorted(
        list(CHECKPOINT_DIR.glob(f"{CHECKPOINT_PREFIX}*.pth")) +
        list(CHECKPOINT_DIR.glob(f"{CHECKPOINT_PREFIX}*.pt"))
    )

    if not checkpoints:
        print("Nenhum checkpoint encontrado.")
        return

    print(
        f"{'Checkpoint':<35}"
        f"{'Tamanho (MB)':>15}"
        f"{'Parâmetros':>20}"
        f"{'Parâmetros (M)':>18}"
    )
    print("-" * 90)

    for checkpoint_path in checkpoints:

        # Tamanho do arquivo
        size_mb = checkpoint_path.stat().st_size / (1024 ** 2)

        try:
            checkpoint = torch.load(
                checkpoint_path,
                map_location="cpu",
                weights_only=False
            )

            # -------------------------------------------------
            # Caso 1: checkpoint é diretamente um nn.Module
            # -------------------------------------------------
            if isinstance(checkpoint, torch.nn.Module):

                total_params = sum(
                    p.numel()
                    for p in checkpoint.parameters()
                )

            # -------------------------------------------------
            # Caso 2: checkpoint é um dicionário
            # -------------------------------------------------
            elif isinstance(checkpoint, dict):

                # Possíveis nomes usados para o modelo
                state_dict = None

                if "state_dict" in checkpoint:
                    state_dict = checkpoint["state_dict"]

                elif "model_state_dict" in checkpoint:
                    state_dict = checkpoint["model_state_dict"]

                elif "model" in checkpoint:

                    model = checkpoint["model"]

                    if isinstance(model, torch.nn.Module):
                        total_params = sum(
                            p.numel()
                            for p in model.parameters()
                        )
                        state_dict = None

                    elif isinstance(model, dict):
                        state_dict = model

                # Se o próprio checkpoint já for um state_dict
                elif all(
                    isinstance(v, torch.Tensor)
                    for v in checkpoint.values()
                ):
                    state_dict = checkpoint

                if state_dict is not None:
                    total_params = count_parameters_from_state_dict(
                        state_dict
                    )

                elif "total_params" not in locals():
                    print(
                        f"{checkpoint_path.name}: "
                        "não foi possível identificar os parâmetros."
                    )
                    continue

            else:
                print(
                    f"{checkpoint_path.name}: "
                    "formato não reconhecido."
                )
                continue

            print(
                f"{checkpoint_path.name:<35}"
                f"{size_mb:>15.2f}"
                f"{total_params:>20,}"
                f"{total_params / 1e6:>18.2f}"
            )

            # Limpa variável para a próxima iteração
            if "total_params" in locals():
                del total_params

        except Exception as e:
            print(
                f"{checkpoint_path.name}: erro ao carregar - {e}"
            )


if __name__ == "__main__":
    get_checkpoint_info()

Checkpoint                            Tamanho (MB)          Parâmetros    Parâmetros (M)
------------------------------------------------------------------------------------------
model_final.pth                              50.81          13,317,888             13.32
model_final_stage0.pth                       25.52           6,687,616              6.69
model_final_stage1.pth                       25.30           6,630,272              6.63
